In [ ]:
"""simulators_service — synthetic Mongo docs for the eod_sale_service pipeline
(Pay Bill / Pay Card / Top Up).

Mirrors the real source topology: three MongoDB collections, each a bill document with
a nested items array + an orderInfo object. Unlike eod_sale_product, transactionTime is
VN-LOCAL (the POS writes local time), so report_date is a plain date-part — no +7h shift.

Deterministic: per-day seed -> reproducible yet distinct per day; ids globally unique
across days (date-tagged). quantity is always 1 (a service txn is one line).

Usage (Fabric):
    %run simulators_service.py
    save_service_bronze(spark, "2026-07-20")                 # one day -> bronze.*
    save_service_bronze(spark, start="2026-07-01", end="2026-07-07")   # a range
"""
import datetime as dt
import functools
import json
import random

STORES = [("1031", "Millenium Masteri D4"), ("1164", "246 Bui Vien D1"),
          ("1078", "S5.01-SH10 VGP THD"), ("1200", "Test Store Alpha")]
SUPPLIER = ("3930239", "CONG TY CO PHAN DICH VU TRUC TUYEN CONG DONG VIET")

# Per-service profile (from prod ClickHouse): Pay Bill = commission service (Z07,
# movement 22, amount in amountBeforeVat, orderInfo populated); Pay Card / Top Up =
# cost services (Z08, movement 20, amount in amountVat, purchase price set, no commission).
SERVICES = {
    "Pay Bill": dict(movement=22, order_type="PayBill", rbt="Z07 - Service Commission",
                     commission=True, purchase_price=False, n_per_day=40,
                     providers=["EVNSG", "EVNHN", "SAWACO"],
                     products=[("97003328", "EVN Ho Chi Minh", "DIEN")],
                     amount_range=(200_000, 3_000_000)),
    "Pay Card": dict(movement=20, order_type="PayCode", rbt="Z08 - Service with Cost",
                     commission=False, purchase_price=True, n_per_day=200,
                     providers=["Viettel", "VINAPHONE", "MOBIFONE"],
                     products=[("96003383", "The cao Viettel 50.000", "")],
                     amount_range=(20_000, 500_000)),
    "Top Up": dict(movement=20, order_type="PayCode", rbt="Z08 - Service with Cost",
                   commission=False, purchase_price=True, n_per_day=5,
                   providers=["VINAPHONE", "Viettel"],
                   products=[("96003403", "TOPUP VinaPhone 500.000", "")],
                   amount_range=(50_000, 500_000)),
}
COLL_OF = {"Pay Bill": "pay_bill_transaction", "Pay Card": "pay_card_transaction",
           "Top Up": "top_up_transaction"}


In [ ]:
def _parse_date(s):
    return dt.date.fromisoformat(str(s)[:10])


def _daterange(start, end=None):
    d0, d1 = _parse_date(start), _parse_date(end or start)
    return [(d0 + dt.timedelta(days=i)).isoformat() for i in range((d1 - d0).days + 1)]


@functools.lru_cache(maxsize=None)
def build_service_docs(day):
    """Native Mongo docs (nested items + orderInfo) for one VN day, keyed by collection."""
    d = _parse_date(day)
    tag = d.strftime("%y%m%d")
    rnd = random.Random(77_000_000 + d.toordinal())
    docs = {c: [] for c in COLL_OF.values()}
    seq = 0
    for svc, cfg in SERVICES.items():
        for _ in range(cfg["n_per_day"]):
            seq += 1
            tt = dt.datetime(d.year, d.month, d.day,
                             rnd.choices(range(8, 21))[0], rnd.randint(0, 59), rnd.randint(0, 59))
            store = rnd.choice(STORES)
            amount = float(rnd.randint(*cfg["amount_range"]) // 1000 * 1000)
            pid, pname, svc_id = rnd.choice(cfg["products"])
            provider = rnd.choice(cfg["providers"])
            pm = rnd.choice([0, 4]) if svc != "Top Up" else 0
            commission = round(amount * 0.008) if cfg["commission"] else 0
            pp_tax = round(amount * 0.983) if cfg["purchase_price"] else 0
            pp_notax = round(pp_tax / 1.1) if cfg["purchase_price"] else 0
            item = {"productCode": pid, "productName": pname, "productUomId": str(4000 + seq % 200),
                    "supplierCode": SUPPLIER[0], "supplierName": SUPPLIER[1], "quantity": 1,
                    "purchasePriceWithTax": pp_tax, "purchasePriceWithoutTax": pp_notax,
                    "totalAmount": amount, "retailBusinessType": cfg["rbt"],
                    "commissionOnVnd": round(commission * 0.9) if cfg["commission"] else 0,
                    "totalCommission": commission}
            order_info = {"orderNo": f"PY{tag}{seq:06d}{provider[:4].upper()}",
                          "serviceId": (svc_id if svc == "Pay Bill" else ""),
                          "providerId": provider,
                          "customerId": (f"PE{seq:011d}" if svc == "Pay Bill" else ""),
                          "orderType": cfg["order_type"]}
            docs[COLL_OF[svc]].append({
                "transactionId": f"{store[0]}{tag}{seq:07d}", "transactionTime": tt, "postingTime": tt,
                "storeId": store[0], "storeName": store[1], "storeType": "CORPORATE",
                "movementType": cfg["movement"],
                "hasInvoice": bool(rnd.random() < 0.1 and svc != "Pay Bill"),
                "customerAgeRange": rnd.choice(["0", "2", "3", "5"]),
                "customerNationality": rnd.choice([0, 1, 2, 3]), "customerGender": rnd.choice([1, 2, 0]),
                "edcType": 0, "cash": int(amount), "serviceName": svc,
                "amountBeforeVat": int(amount if svc == "Pay Bill" else 0),
                "amountVat": int(0 if svc == "Pay Bill" else amount), "totalAmountWithoutPromotion": 0,
                "staffId": f"10{rnd.randint(10000, 99999)}", "staffName": "Staff Member",
                "posId": f"{store[0]}01", "paymentMethod": pm, "paymentMethodValue": 100,
                "transactionRecordStartTime": tt, "transactionRecordEndTime": tt,
                "promotionTotalAmount": 1, "totalProductPromotionAmountWithoutTax": 0,
                "totalVoucherPromotionAmountWithoutTax": 0, "totalBillPromotionAmountWithoutTax": 0,
                "serviceFee": 0, "customerServiceFee": 0, "skipValidateOver50Percent": False,
                "receivableAmount": 0, "prePaid": False,
                "syncedSapStatus": rnd.choice(["NOT_SYNC", "SYNCED"]), "syncedSapTime": None,
                "orderInfo": order_info, "items": [item]})
    return docs


In [ ]:
# Top-level scalar fields (nested items/orderInfo are stored as JSON strings in bronze,
# exactly as a Copy activity or the Mongo Spark connector would land them).
_TOPLEVEL = [("transactionId", "str"), ("transactionTime", "ts"), ("postingTime", "ts"),
             ("storeId", "str"), ("storeName", "str"), ("storeType", "str"),
             ("movementType", "long"), ("hasInvoice", "bool"), ("customerAgeRange", "str"),
             ("customerNationality", "long"), ("customerGender", "long"), ("edcType", "long"),
             ("cash", "long"), ("serviceName", "str"), ("amountBeforeVat", "long"),
             ("amountVat", "long"), ("totalAmountWithoutPromotion", "long"), ("staffId", "str"),
             ("staffName", "str"), ("posId", "str"), ("paymentMethod", "long"),
             ("paymentMethodValue", "long"), ("transactionRecordStartTime", "ts"),
             ("transactionRecordEndTime", "ts"), ("promotionTotalAmount", "long"),
             ("totalProductPromotionAmountWithoutTax", "long"),
             ("totalVoucherPromotionAmountWithoutTax", "long"),
             ("totalBillPromotionAmountWithoutTax", "long"), ("serviceFee", "long"),
             ("customerServiceFee", "long"), ("skipValidateOver50Percent", "bool"),
             ("receivableAmount", "long"), ("prePaid", "bool"), ("syncedSapStatus", "str"),
             ("syncedSapTime", "ts")]


def _bronze_struct():
    from pyspark.sql.types import (StructType, StructField, StringType, LongType,
                                    TimestampType, BooleanType)
    tt = {"str": StringType(), "long": LongType(), "ts": TimestampType(), "bool": BooleanType()}
    return StructType([StructField(c, tt[t]) for c, t in _TOPLEVEL] +
                      [StructField("orderInfo", StringType()), StructField("items", StringType())])


def save_service_bronze(spark, start, end=None, schema="bronze", fmt="delta"):
    """Write the 3 collections straight to bronze for the day range (nested -> JSON string).
    Run the pipeline with with_ingest=False afterwards."""
    st = _bronze_struct()
    acc = {c: [] for c in COLL_OF.values()}
    for day in _daterange(start, end):
        for coll, rows in build_service_docs(day).items():
            acc[coll] += rows
    for coll, rows in acc.items():
        tuples = [tuple([r[c] for c, _ in _TOPLEVEL] +
                        [json.dumps(r["orderInfo"]), json.dumps(r["items"])]) for r in rows]
        w = spark.createDataFrame(tuples, st).write.format(fmt).mode("overwrite")
        if fmt == "delta":
            w = w.option("overwriteSchema", "true")
        w.saveAsTable(f"{schema}.{coll}")
        print(f"{schema}.{coll}: {len(rows)} rows")
